In [83]:
from importlib import reload
import sys
from pathlib import Path

src_path = (Path.cwd().parent / "src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import dictionary_match
import embedder
import score
import title_similarity
dictionary_match = reload(dictionary_match)
embedder = reload(embedder)
score = reload(score)
title_similarity = reload(title_similarity)

from dictionary_match import compute_dictionary_pairs_and_topk, dumps_skill_columns
from dictionary_match import to_skill_list

from embedder import TextEmbedder, compute_text_pairs_and_topk

from score import compute_fused_pairs_and_topk, save_model_scores, compute_final_scores, compute_ranks_and_topk, get_topk_for_resume

from title_similarity import compute_title_pairs_and_topk

import json
import pandas as pd

processed_dir = "../data/processed"

In [39]:
resume_ranked_path = f"{processed_dir}/resume_skills_extracted.csv"
job_ranked_path = f"{processed_dir}/job_skills_extracted.csv"

resume_out = pd.read_csv(resume_ranked_path)
job_out = pd.read_csv(job_ranked_path)
print("Loaded ranked extracted files")

Loaded ranked extracted files


# TF-IDF

In [40]:
embedder = "tfidf"

model_pairs_df, model_topk_df = compute_text_pairs_and_topk(
    resume_df=resume_out,
    job_df=job_out,
    model_name=embedder,
)

In [41]:
model_pairs_df = model_pairs_df.rename(columns={
    "final_score": "tfidf_score"
})

tfidf_scores_df = save_model_scores(
    df=model_pairs_df,
    output_path="../data/processed/tfidf_scores.csv"
)

Saved to ../data/processed/tfidf_scores.csv


In [42]:
display(model_topk_df.head())
display(model_pairs_df.head())

,resume_id,resume_idx,job_idx,job_title,company,job_location,model_name,semantic_similarity,final_score,rank
0,20345168,0,715,Payroll Manager,MetroMSR - Staffing Resources,"Alexandria, VA",tfidf,0.2336,23.36,1
1,20345168,0,1445,Senior Accountant Tax,Paragon Accountants,"San Diego, CA",tfidf,0.2022,20.22,2
2,20345168,0,2937,***Accounting Senior Manager | Hybrid in Tampa...,Vaco,"Tampa, FL",tfidf,0.1968,19.68,3
3,20345168,0,808,Controller,Chesapeake Contracting Group,"Baltimore, MD",tfidf,0.1935,19.35,4
4,20345168,0,2099,Senior Accountant,Jobot,"Los Angeles, CA",tfidf,0.1885,18.85,5


,resume_id,resume_idx,job_idx,job_title,company,job_location,model_name,semantic_similarity,tfidf_score
0,20345168,0,0,Registered Nurse (RN) at Aveanna,Health eCareers,"Vancouver, WA",tfidf,0.0118,1.18
1,20345168,0,1,Locum | Physician Obstetrics and Gynecology,Weatherby Healthcare,"New Richmond, WI",tfidf,0.0100,1.00
2,20345168,0,2,Technology Sales (Recruiting) Business Develop...,Proven Recruiting,"Austin, TX",tfidf,0.0270,2.70
3,20345168,0,3,Travel RN Oncology 1862.00/week - 23731031EXPP...,TravelNurseSource,"Lexington, KY",tfidf,0.0080,0.80
4,20345168,0,4,Floor Supervisor Full Time - TOMMY HILFIGER - ...,Tommy Hilfiger,"Las Vegas, NV",tfidf,0.0375,3.75


# BGE-SMALL

In [43]:
embedder = "BAAI/bge-small-en-v1.5"

model_pairs_df, model_topk_df = compute_text_pairs_and_topk(
    resume_df=resume_out,
    job_df=job_out,
    model_name=embedder,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4333.99it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [44]:
model_pairs_df = model_pairs_df.rename(columns={
    "final_score": "bge_score"
})

bge_scores_df = save_model_scores(
    df=model_pairs_df,
    output_path="../data/processed/bge_scores.csv"
)

Saved to ../data/processed/bge_scores.csv


In [45]:
display(model_topk_df.head())
display(model_pairs_df.head())

,resume_id,resume_idx,job_idx,job_title,company,job_location,model_name,semantic_similarity,final_score,rank
0,20345168,0,698,Sr. Staff Accountant,Robert Half,"Avondale, PA",BAAI/bge-small-en-v1.5,0.8165,81.65,1
1,20345168,0,1093,Staff Accountant,Accounting Career Consultants & HR Career Cons...,"St Louis, MO",BAAI/bge-small-en-v1.5,0.8112,81.12,2
2,20345168,0,280,Staff Accountant,Willamette Falls Paper Company,"West Linn, OR",BAAI/bge-small-en-v1.5,0.8070,80.70,3
3,20345168,0,715,Payroll Manager,MetroMSR - Staffing Resources,"Alexandria, VA",BAAI/bge-small-en-v1.5,0.7959,79.59,4
4,20345168,0,1445,Senior Accountant Tax,Paragon Accountants,"San Diego, CA",BAAI/bge-small-en-v1.5,0.7919,79.19,5


,resume_id,resume_idx,job_idx,job_title,company,job_location,model_name,semantic_similarity,bge_score
0,20345168,0,0,Registered Nurse (RN) at Aveanna,Health eCareers,"Vancouver, WA",BAAI/bge-small-en-v1.5,0.6605,66.05
1,20345168,0,1,Locum | Physician Obstetrics and Gynecology,Weatherby Healthcare,"New Richmond, WI",BAAI/bge-small-en-v1.5,0.6696,66.96
2,20345168,0,2,Technology Sales (Recruiting) Business Develop...,Proven Recruiting,"Austin, TX",BAAI/bge-small-en-v1.5,0.6966,69.66
3,20345168,0,3,Travel RN Oncology 1862.00/week - 23731031EXPP...,TravelNurseSource,"Lexington, KY",BAAI/bge-small-en-v1.5,0.6906,69.06
4,20345168,0,4,Floor Supervisor Full Time - TOMMY HILFIGER - ...,Tommy Hilfiger,"Las Vegas, NV",BAAI/bge-small-en-v1.5,0.6944,69.44


# MiniLM


In [46]:
embedder = "sentence-transformers/all-MiniLM-L6-v2"

model_pairs_df, model_topk_df = compute_text_pairs_and_topk(
    resume_df=resume_out,
    job_df=job_out,
    model_name=embedder,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11183.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [47]:
model_pairs_df = model_pairs_df.rename(columns={
    "final_score": "minilm_score"
})

minilm_scores_df = save_model_scores(
    df=model_pairs_df,
    output_path="../data/processed/minilm_scores.csv"
)

Saved to ../data/processed/minilm_scores.csv


In [48]:
display(model_topk_df.head())
display(model_pairs_df.head())

,resume_id,resume_idx,job_idx,job_title,company,job_location,model_name,semantic_similarity,final_score,rank
0,20345168,0,1093,Staff Accountant,Accounting Career Consultants & HR Career Cons...,"St Louis, MO",sentence-transformers/all-MiniLM-L6-v2,0.7574,75.74,1
1,20345168,0,698,Sr. Staff Accountant,Robert Half,"Avondale, PA",sentence-transformers/all-MiniLM-L6-v2,0.7345,73.45,2
2,20345168,0,1445,Senior Accountant Tax,Paragon Accountants,"San Diego, CA",sentence-transformers/all-MiniLM-L6-v2,0.7325,73.25,3
3,20345168,0,2436,Sr. Accountant,Robert Half,"Atlanta, GA",sentence-transformers/all-MiniLM-L6-v2,0.7091,70.91,4
4,20345168,0,832,Audit & Accounts Senior,Public Practice Recruitment Ltd - Experts in P...,"Cannock, England, United Kingdom",sentence-transformers/all-MiniLM-L6-v2,0.7086,70.86,5


,resume_id,resume_idx,job_idx,job_title,company,job_location,model_name,semantic_similarity,minilm_score
0,20345168,0,0,Registered Nurse (RN) at Aveanna,Health eCareers,"Vancouver, WA",sentence-transformers/all-MiniLM-L6-v2,0.2781,27.81
1,20345168,0,1,Locum | Physician Obstetrics and Gynecology,Weatherby Healthcare,"New Richmond, WI",sentence-transformers/all-MiniLM-L6-v2,0.3104,31.04
2,20345168,0,2,Technology Sales (Recruiting) Business Develop...,Proven Recruiting,"Austin, TX",sentence-transformers/all-MiniLM-L6-v2,0.3407,34.07
3,20345168,0,3,Travel RN Oncology 1862.00/week - 23731031EXPP...,TravelNurseSource,"Lexington, KY",sentence-transformers/all-MiniLM-L6-v2,0.1924,19.24
4,20345168,0,4,Floor Supervisor Full Time - TOMMY HILFIGER - ...,Tommy Hilfiger,"Las Vegas, NV",sentence-transformers/all-MiniLM-L6-v2,0.3670,36.70


# MPNet

In [49]:
embedder = "sentence-transformers/all-mpnet-base-v2"

model_pairs_df, model_topk_df = compute_text_pairs_and_topk(
    resume_df=resume_out,
    job_df=job_out,
    model_name=embedder,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3349.84it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [50]:
model_pairs_df = model_pairs_df.rename(columns={
    "final_score": "mpnet_score"
    })

mpnet_scores_df = save_model_scores(
    df=model_pairs_df,
    output_path="../data/processed/mpnet_scores.csv"
)

Saved to ../data/processed/mpnet_scores.csv


In [51]:
display(model_topk_df.head())
display(model_pairs_df.head())

,resume_id,resume_idx,job_idx,job_title,company,job_location,model_name,semantic_similarity,final_score,rank
0,20345168,0,280,Staff Accountant,Willamette Falls Paper Company,"West Linn, OR",sentence-transformers/all-mpnet-base-v2,0.8409,84.09,1
1,20345168,0,2099,Senior Accountant,Jobot,"Los Angeles, CA",sentence-transformers/all-mpnet-base-v2,0.8409,84.09,2
2,20345168,0,2731,Accountant/Staff Accountant,Intellectt Inc,"Princeton, NJ",sentence-transformers/all-mpnet-base-v2,0.8402,84.02,3
3,20345168,0,1445,Senior Accountant Tax,Paragon Accountants,"San Diego, CA",sentence-transformers/all-mpnet-base-v2,0.8075,80.75,4
4,20345168,0,1093,Staff Accountant,Accounting Career Consultants & HR Career Cons...,"St Louis, MO",sentence-transformers/all-mpnet-base-v2,0.8046,80.46,5


,resume_id,resume_idx,job_idx,job_title,company,job_location,model_name,semantic_similarity,mpnet_score
0,20345168,0,0,Registered Nurse (RN) at Aveanna,Health eCareers,"Vancouver, WA",sentence-transformers/all-mpnet-base-v2,0.4219,42.19
1,20345168,0,1,Locum | Physician Obstetrics and Gynecology,Weatherby Healthcare,"New Richmond, WI",sentence-transformers/all-mpnet-base-v2,0.2469,24.69
2,20345168,0,2,Technology Sales (Recruiting) Business Develop...,Proven Recruiting,"Austin, TX",sentence-transformers/all-mpnet-base-v2,0.4211,42.11
3,20345168,0,3,Travel RN Oncology 1862.00/week - 23731031EXPP...,TravelNurseSource,"Lexington, KY",sentence-transformers/all-mpnet-base-v2,0.3238,32.38
4,20345168,0,4,Floor Supervisor Full Time - TOMMY HILFIGER - ...,Tommy Hilfiger,"Las Vegas, NV",sentence-transformers/all-mpnet-base-v2,0.4452,44.52


# Title Similarity

In [53]:
title_pairs_df, title_topk_df = compute_title_pairs_and_topk(
    resume_df=resume_out,
    job_df=job_out,
    resume_title_col="Category",
    job_title_col="job_title",
    top_k=5,
)

In [66]:
title_pairs_df = title_pairs_df.rename(columns={
    "final_score": "title_score"
})

title_scores_df = save_model_scores(
    df=title_pairs_df,
    output_path="../data/processed/title_scores.csv"
)

Saved to ../data/processed/title_scores.csv


In [67]:
display(title_topk_df.head())
display(title_pairs_df.head())

,resume_id,resume_idx,resume_title,job_idx,job_title,company,job_location,job_link,title_similarity,final_score,rank
0,20345168,0,ACCOUNTANT,1195,Accountant,"Coolray Heating, Cooling, Plumbing, Electrical","Birmingham, AL",https://www.linkedin.com/jobs/view/accountant-...,1.0,100.0,1
1,20345168,0,ACCOUNTANT,1404,Accountant,Highmark Inc.,"Pennsylvania, United States",https://www.linkedin.com/jobs/view/accountant-...,1.0,100.0,2
2,20345168,0,ACCOUNTANT,2677,Accountant,Kassen Recruitment,"Markham, Ontario, Canada",https://ca.linkedin.com/jobs/view/accountant-a...,1.0,100.0,3
3,20345168,0,ACCOUNTANT,157,Sr Accountant,Golden Nugget,"Las Vegas, NV",https://www.linkedin.com/jobs/view/sr-accounta...,0.9,90.0,4
4,20345168,0,ACCOUNTANT,280,Staff Accountant,Willamette Falls Paper Company,"West Linn, OR",https://www.linkedin.com/jobs/view/staff-accou...,0.9,90.0,5


,resume_id,resume_idx,resume_title,job_idx,job_title,company,job_location,job_link,title_similarity,title_score
0,20345168,0,ACCOUNTANT,0,Registered Nurse (RN) at Aveanna,Health eCareers,"Vancouver, WA",https://www.linkedin.com/jobs/view/registered-...,0.1905,19.05
1,20345168,0,ACCOUNTANT,1,Locum | Physician Obstetrics and Gynecology,Weatherby Healthcare,"New Richmond, WI",https://www.linkedin.com/jobs/view/locum-physi...,0.1509,15.09
2,20345168,0,ACCOUNTANT,2,Technology Sales (Recruiting) Business Develop...,Proven Recruiting,"Austin, TX",https://www.linkedin.com/jobs/view/technology-...,0.2059,20.59
3,20345168,0,ACCOUNTANT,3,Travel RN Oncology 1862.00/week - 23731031EXPP...,TravelNurseSource,"Lexington, KY",https://www.linkedin.com/jobs/view/travel-rn-o...,0.1356,13.56
4,20345168,0,ACCOUNTANT,4,Floor Supervisor Full Time - TOMMY HILFIGER - ...,Tommy Hilfiger,"Las Vegas, NV",https://www.linkedin.com/jobs/view/floor-super...,0.1149,11.49


# Final Hybrid Scores

In [68]:
# Read in all the scores for final fusion
skill_pairs_df = f"{processed_dir}/dictionary_pair_scores.csv"

skill_scores_df = pd.read_csv(skill_pairs_df)
print("Loaded skill dictionary scores from extracted files")

Loaded skill dictionary scores from extracted files


In [69]:
# Sanity check to ensure all resume job pairs are same across all models
print(title_scores_df.shape)
print(tfidf_scores_df.shape)
print(bge_scores_df.shape)
print(minilm_scores_df.shape)
print(mpnet_scores_df.shape)
print(skill_scores_df.shape)

(2136000, 10)
(2136000, 9)
(2136000, 9)
(2136000, 9)
(2136000, 9)
(2136000, 14)


In [76]:
#---------------- Final fusion of all scores for TF-IDF -------------------
skill_weight = 0.3
model_weight = 0.5
title_weight = 0.2

tfidf_final_score_df = compute_final_scores(
    skill_pairs_df=skill_scores_df,
    model_pairs_df=tfidf_scores_df,
    title_pairs_df=title_scores_df,
    skill_weight=skill_weight,
    model_weight=model_weight,
    title_weight=title_weight,
    output_path="../data/scores/tfidf_final_scores.csv"
)
print("Computed final fused scores for TF-IDF model")

#---------------- Final fusion of all scores for BGE-SMALL -------------------
bge_final_score_df = compute_final_scores(
    skill_pairs_df=skill_scores_df,
    model_pairs_df=bge_scores_df,
    title_pairs_df=title_scores_df,
    skill_weight=skill_weight,
    model_weight=model_weight,
    title_weight=title_weight,
    output_path="../data/scores/bge_final_scores.csv"
)
print("Computed final fused scores for BGE-SMALL model")

#---------------- Final fusion of all scores for MPNET -------------------
mpnet_final_score_df = compute_final_scores(
    skill_pairs_df=skill_scores_df,
    model_pairs_df=mpnet_scores_df,
    title_pairs_df=title_scores_df,
    skill_weight=skill_weight,
    model_weight=model_weight,
    title_weight=title_weight,
    output_path="../data/scores/mpnet_final_scores.csv"
)
print("Computed final fused scores for MPNET model")

#---------------- Final fusion of all scores for MiniLM -------------------
minilm_final_score_df = compute_final_scores(
    skill_pairs_df=skill_scores_df,
    model_pairs_df=minilm_scores_df,
    title_pairs_df=title_scores_df,
    skill_weight=skill_weight,
    model_weight=model_weight,
    title_weight=title_weight,
    output_path="../data/scores/minilm_final_scores.csv"
)
print("Computed final fused scores for MiniLM model")

Final scores saved to ../data/processed/tfidf_final_scores.csv
Computed final fused scores for TF-IDF model
Final scores saved to ../data/processed/bge_final_scores.csv
Computed final fused scores for BGE-SMALL model
Final scores saved to ../data/processed/mpnet_final_scores.csv
Computed final fused scores for MPNET model
Final scores saved to ../data/processed/minilm_final_scores.csv
Computed final fused scores for MiniLM model


In [81]:
tfidf_ranked_df, tfidf_topk_df = compute_ranks_and_topk(
    final_scores_df=tfidf_final_score_df,
    score_col="final_score",
    resume_id_col="resume_id",
    top_k=5
)

top5_resume1 = get_topk_for_resume(tfidf_ranked_df, resume_id=20345168, k=5)
top5_resume1

Full ranked results saved to ../data/processed/final_ranked.csv
Top-5 results saved to ../data/processed/final_topk.csv


,resume_id,resume_idx,resume_category,job_idx,job_link,job_title,company,job_location,dictionary_score,job_coverage,skill_jaccard,matched_skills,missing_skills,extra_skills,semantic_similarity,title_similarity,final_score,rank
1445,20345168,0,ACCOUNTANT,1445,https://www.linkedin.com/jobs/view/senior-acco...,Senior Accountant Tax,Paragon Accountants,"San Diego, CA",15.20,0.1964,0.0485,"[""business management"", ""deadlines"", ""excel"", ...","[""accounting"", ""advanced"", ""advanced excel"", ""...","['accountant', 'accounting duties', 'accountin...",0.2022,0.90,32.670,1
1759,20345168,0,ACCOUNTANT,1759,https://www.linkedin.com/jobs/view/accounting-...,Accounting Manager,MDC Partners,"Los Angeles, CA",23.00,0.3023,0.0613,"[""analysis"", ""billing"", ""excel"", ""outlook"", ""p...","[""account payable"", ""account receivable"", ""acc...","['accountant', 'accounting duties', 'accountin...",0.1651,0.85,32.155,2
698,20345168,0,ACCOUNTANT,698,https://www.linkedin.com/jobs/view/sr-staff-ac...,Sr. Staff Accountant,Robert Half,"Avondale, PA",16.27,0.2162,0.0379,"[""analysis"", ""excel"", ""inventory"", ""ledger"", ""...","[""accounting or finance"", ""accounting reconcil...","['accountant', 'accounting duties', 'accountin...",0.1679,0.90,31.276,3
2937,20345168,0,ACCOUNTANT,2937,https://www.linkedin.com/jobs/view/accounting-...,***Accounting Senior Manager | Hybrid in Tampa...,Vaco,"Tampa, FL",12.92,0.1707,0.0324,"[""excel"", ""inventory"", ""payable"", ""policies"", ...","[""accounting policies"", ""accounts payable"", ""a...","['accountant', 'accounting duties', 'accountin...",0.1968,0.85,30.716,4
1195,20345168,0,ACCOUNTANT,1195,https://www.linkedin.com/jobs/view/accountant-...,Accountant,"Coolray Heating, Cooling, Plumbing, Electrical","Birmingham, AL",11.15,0.1458,0.0314,"[""analysis"", ""excel"", ""ledger"", ""policies"", ""q...","[""accounting policies"", ""accounting procedures...","['accountant', 'accounting duties', 'accountin...",0.1330,1.00,29.995,5


In [82]:
bge_ranked_df, bge_topk_df = compute_ranks_and_topk(
    final_scores_df=bge_final_score_df,
    score_col="final_score",
    resume_id_col="resume_id",
    top_k=5
)

top5_resume1 = get_topk_for_resume(
    bge_ranked_df[
        ["resume_id", "resume_category", "job_title",
         "dictionary_score", "semantic_similarity",
         "title_similarity", "final_score", "rank"]
    ],
    resume_id=20345168,
    k=5
)
top5_resume1

Full ranked results saved to ../data/processed/final_ranked.csv
Top-5 results saved to ../data/processed/final_topk.csv


,resume_id,resume_category,job_title,dictionary_score,semantic_similarity,title_similarity,final_score,rank
698,20345168,ACCOUNTANT,Sr. Staff Accountant,16.27,0.8165,0.90,63.706,1
280,20345168,ACCOUNTANT,Staff Accountant,17.12,0.8070,0.90,63.486,2
1759,20345168,ACCOUNTANT,Accounting Manager,23.00,0.7845,0.85,63.125,3
1445,20345168,ACCOUNTANT,Senior Accountant Tax,15.20,0.7919,0.90,62.155,4
1188,20345168,ACCOUNTANT,Property Accounting Supervisor,21.36,0.7739,0.85,62.103,5
